# 服装视觉聚类 → 爆款方向分析
Pipeline: HuggingFace `tonyassi/clothing-sales-data-embeddings` → UMAP 降维 → HDBSCAN 聚类 → 导出 cluster JSON

In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd

print('Loading dataset...')
ds = load_dataset('tonyassi/clothing-sales-data-embeddings', split='train')
df = ds.to_pandas()
print(f'Loaded {len(df)} rows')
print(df.columns.tolist())
df.head(3)

In [ ]:
# ── Inspect embedding column ──────────────────────────────────────────────────
sample_emb = df['embeddings'].iloc[0]
print(type(sample_emb), len(sample_emb) if hasattr(sample_emb, '__len__') else 'scalar')

# Parse embeddings into matrix
import ast
def parse_emb(x):
    if isinstance(x, (list, np.ndarray)):
        return np.array(x, dtype=np.float32)
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=np.float32)
    return np.array(x, dtype=np.float32)

X = np.vstack(df['embeddings'].apply(parse_emb).values)
print(f'Embedding matrix: {X.shape}')

In [ ]:
# ── Check other columns ───────────────────────────────────────────────────────
print(df[['title','price','units_sold']].head(10).to_string())
print('\nPrice range:', df['price'].min(), '–', df['price'].max())
print('Units sold range:', df['units_sold'].min(), '–', df['units_sold'].max())

In [ ]:
# ── UMAP: 768D → 10D (for HDBSCAN) + 2D (for viz) ────────────────────────────
import umap
print('UMAP 10D...')
reducer_10d = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.0,
                         metric='cosine', random_state=42, verbose=True)
X_10d = reducer_10d.fit_transform(X)

print('UMAP 2D...')
reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=42)
X_2d = reducer_2d.fit_transform(X)
print('Done')

In [ ]:
# ── HDBSCAN clustering ────────────────────────────────────────────────────────
import hdbscan
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(X_10d)

df['cluster'] = labels
df['x2d'] = X_2d[:, 0]
df['y2d'] = X_2d[:, 1]

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).mean() * 100
print(f'Clusters: {n_clusters}  |  Noise: {noise_pct:.1f}%')
print(pd.Series(labels).value_counts().head(15))

In [ ]:
# ── Visualize clusters ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(10, 8))
colors = cm.tab20(np.linspace(0, 1, max(n_clusters, 1)))

noise = df[df['cluster'] == -1]
ax.scatter(noise['x2d'], noise['y2d'], c='lightgray', s=8, alpha=0.3, label='noise')

for i, cid in enumerate(sorted([c for c in df['cluster'].unique() if c != -1])):
    sub = df[df['cluster'] == cid]
    ax.scatter(sub['x2d'], sub['y2d'], s=20, alpha=0.7,
               label=f'C{cid} (n={len(sub)})')
    cx, cy = sub['x2d'].mean(), sub['y2d'].mean()
    ax.annotate(f'C{cid}', (cx, cy), fontsize=9, ha='center', fontweight='bold')

ax.set_title('UMAP 2D + HDBSCAN Clusters (clothing embeddings)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.savefig('cluster_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cluster_viz.png')

In [ ]:
# ── Per-cluster stats ─────────────────────────────────────────────────────────
cluster_df = df[df['cluster'] != -1].copy()

stats = cluster_df.groupby('cluster').agg(
    count=('title', 'count'),
    avg_price=('price', 'mean'),
    avg_sales=('units_sold', 'mean'),
    total_sales=('units_sold', 'sum'),
    top_title=('title', lambda x: x.iloc[cluster_df.loc[x.index, 'units_sold'].argmax()])
).sort_values('avg_sales', ascending=False)

print(stats.to_string())

In [ ]:
# ── Top products per cluster ──────────────────────────────────────────────────
for cid in stats.head(6).index:
    sub = cluster_df[cluster_df['cluster'] == cid].nlargest(5, 'units_sold')
    print(f'\n── Cluster {cid} (n={len(cluster_df[cluster_df["cluster"]==cid])}, avg_sales={stats.loc[cid,"avg_sales"]:.0f}) ──')
    for _, row in sub.iterrows():
        print(f'  [{row["units_sold"]}] ${row["price"]:.2f} | {row["title"][:80]}')

In [ ]:
# ── Auto-label clusters with a simple keyword extractor ──────────────────────
from collections import Counter
import re

STOPWORDS = {'dress','women','womens','ladies','summer','casual','plus','size',
             'short','long','sleeve','the','and','for','with','midi','maxi',
             'mini','a','in','of','to','at','by','from','on','style','fashion'}

def top_keywords(titles, n=5):
    words = []
    for t in titles:
        toks = re.findall(r'[a-zA-Z]+', str(t).lower())
        words += [w for w in toks if w not in STOPWORDS and len(w) > 2]
    return [w for w, _ in Counter(words).most_common(n)]

cluster_labels = {}
for cid in sorted(cluster_df['cluster'].unique()):
    sub = cluster_df[cluster_df['cluster'] == cid]
    kws = top_keywords(sub['title'])
    cluster_labels[cid] = kws
    print(f'C{cid:2d} (n={len(sub):3d}, avg_sales={sub["units_sold"].mean():.0f}): {kws}')

In [ ]:
# ── Pick top 4 clusters by avg_sales, export as JSON for the app ──────────────
import json
from io import BytesIO
from PIL import Image as PILImage
import base64

TOP_N = 4  # we want 4 visual clusters for the app
top_clusters = stats.head(TOP_N).index.tolist()

output_clusters = []

for rank, cid in enumerate(top_clusters):
    sub = cluster_df[cluster_df['cluster'] == cid]
    top_products = sub.nlargest(8, 'units_sold')
    kws = cluster_labels[cid]
    
    # Save top product images as base64 (for mosaic)
    mosaic_b64 = []
    for _, row in top_products.head(4).iterrows():
        try:
            img = row['image']  # PIL Image or bytes
            if isinstance(img, dict) and 'bytes' in img:
                img = PILImage.open(BytesIO(img['bytes']))
            elif isinstance(img, bytes):
                img = PILImage.open(BytesIO(img))
            img = img.convert('RGB').resize((200, 200))
            buf = BytesIO()
            img.save(buf, format='JPEG', quality=70)
            mosaic_b64.append('data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode())
        except Exception as e:
            print(f'Image error: {e}')
    
    cluster_info = {
        'id': int(cid),
        'rank': rank + 1,
        'keywords': kws,
        'name': ' · '.join(kws[:3]),
        'count': int(len(sub)),
        'avg_price': round(float(sub['price'].mean()), 2),
        'avg_sales': round(float(sub['units_sold'].mean()), 1),
        'top_sales': int(sub['units_sold'].max()),
        'mosaic_images': mosaic_b64,  # 4 base64 images
        'top_products': [
            {
                'title': row['title'],
                'price': round(float(row['price']), 2),
                'units_sold': int(row['units_sold']),
                'tags': row.get('tags', ''),
                'rating': float(row['rating']) if 'rating' in row.index else None,
            }
            for _, row in top_products.head(6).iterrows()
        ]
    }
    output_clusters.append(cluster_info)
    print(f'\nCluster {cid}: "{cluster_info["name"]}" | {cluster_info["count"]} items | avg_sales={cluster_info["avg_sales"]} | mosaic_imgs={len(mosaic_b64)}')

out_path = '../lib/cluster-data.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output_clusters, f, ensure_ascii=False, indent=2)

print(f'\nSaved {len(output_clusters)} clusters → {out_path}')

In [ ]:
# ── Also save lightweight version (no base64 images) for inspection ───────────
lite = json.loads(json.dumps(output_clusters))
for c in lite:
    c['mosaic_images'] = [f'[base64 img {i+1}, {len(b)//1024}KB]' for i, b in enumerate(c['mosaic_images'])]

print(json.dumps(lite, indent=2, ensure_ascii=False))

In [ ]:
# ── Demand trend simulation: compare sales of cluster vs overall baseline ─────
# We don't have time-series data, so we use relative avg_sales vs global mean as proxy
global_avg = cluster_df['units_sold'].mean()

for c in output_clusters:
    rel = (c['avg_sales'] / global_avg - 1) * 100
    c['demand_vs_baseline_pct'] = round(rel, 1)
    c['competition'] = '低' if c['count'] < 50 else '中' if c['count'] < 120 else '高'
    print(f'C{c["id"]} "{c["name"]}" → demand {rel:+.1f}% vs baseline | competition: {c["competition"]}')

# Re-save with trend info
with open('../lib/cluster-data.json', 'w', encoding='utf-8') as f:
    json.dump(output_clusters, f, ensure_ascii=False, indent=2)
print('Updated cluster-data.json with demand trend info')